**BERN02 Exercise: Regression**

Name: Yang Shann Wen

Date: 1 September 2026

In [19]:
import pandas as pd
import numpy as np

In [20]:
# Import and load dataset
df = pd.read_csv('/content/pollution_cleaneddata.csv')

In [21]:
# Extract the vector of observations of the predictor: % of families with income < $3000 (POOR)
x_data = df['POOR']

# Extract the vector of observations of the response variable: Total age-adjusted mortality rate per 100,000 (MORT)
y_data = df['MORT']

# Convert into numpy array for calculations
x_data = np.array(x_data)
y_data = np.array(y_data)

In [22]:
# Function for performing predictions with Local Regression, using Gaussian weights
def loess(y, x, k, x0):
  """
  Parameters:
  y: The response variable observations
  x: The predictor observations
  k: The number of nearest neighboring points to include in each local regression
  x0: The target values to be predicted

  Outputs:
  pred: The list of predicted expected values
  se: The list of standard deviations of the expected values

  """

  # Empty list to store my predicted expected values and standard errors
  pred = []
  se = []

  # Run the function for every target value in x0
  for i in x0:

    # Calculate distances between every data point x and the target value
    distances = np.abs(x-i)

    # Pick the k nearest points
    nearest_index = np.argsort(distances)[:k]

    # Fit a weighted OLS line for the local area
    # Extract the x and y values for these k nearest points
    x_neighbors = x[nearest_index]
    y_neighbors = y[nearest_index]


    # A Gaussian weighting function is used to assign greater weights to observations closer to the target value
    # Calculate the distances of the selected observations from i
    distances_neighbors = np.abs(x_neighbors - i)

    # Set the bandwidth to the largest distance among the k neighbours
    bandwidth = np.max(distances_neighbors)

    # Avoid division by zero if all selected x values equal i
    if bandwidth == 0:
      bandwidth = 1e-10

    # Calculate Gaussian weights
    weights = np.exp(-(distances_neighbors**2) / (2 * bandwidth**2))

    # Calculate weighted mean of x
    x_mean = np.sum(weights * x_neighbors) / np.sum(weights)

    # Calculate weighted mean of y
    y_mean = np.sum(weights * y_neighbors) / np.sum(weights)


    # Calculate the slope (beta_1) for the weighted OLS regression
    numerator = np.sum(weights * (x_neighbors-x_mean) * (y_neighbors-y_mean))
    denominator = np.sum(weights * (x_neighbors - x_mean)**2)
    beta_1 = numerator / denominator

    # Calculate the intercept (beta_0)
    beta_0 = y_mean - (beta_1 * x_mean)

    # Plug in target value to make predictions
    y_predicted = beta_0 + (beta_1 * i)
    pred.append(float(np.round(y_predicted, 2)))

    # Calculate the variance of the error (sigma^2)
    # The variance is estimated without weighting, using the unweighted RSS divided by the degrees of freedom (k - 2)
    e = y_neighbors - beta_0 - (beta_1 * x_neighbors)
    variance = (np.sum(e**2)) / (k-2)

    # Calculate the variance of the expected value
    h = ((i - x_mean)**2) / denominator
    variance_expected_value = variance * ((1/k) + h)

    # Square root for standard deviation
    standard_error = np.sqrt(variance_expected_value)
    se.append(float(np.round(standard_error, 2)))

  return pred, se

In [23]:
# Set parameters for prediction
k_values = [5, 10, 15]
x0 = [10, 18, 25]

# Loop through each k value
for k in k_values:
    # Run the function
    pred, se = loess(y_data, x_data, k, x0)

    print(f"\nPredictions when k = {k}:")
    for x_val, p, s in zip(x0, pred, se):
        print(f"  When x0 = {x_val}, MORT = {p}. (Standard error = {s}).")


Predictions when k = 5:
  When x0 = 10, MORT = 873.54. (Standard error = 21.54).
  When x0 = 18, MORT = 969.59. (Standard error = 24.21).
  When x0 = 25, MORT = 1026.34. (Standard error = 29.23).

Predictions when k = 10:
  When x0 = 10, MORT = 899.52. (Standard error = 20.63).
  When x0 = 18, MORT = 957.0. (Standard error = 16.46).
  When x0 = 25, MORT = 1012.95. (Standard error = 26.8).

Predictions when k = 15:
  When x0 = 10, MORT = 901.72. (Standard error = 17.72).
  When x0 = 18, MORT = 958.68. (Standard error = 20.21).
  When x0 = 25, MORT = 1009.47. (Standard error = 24.9).


As seen in the results, the Gaussian-weighted local regression shows a consistent positive association between MORT and POOR, since MORT increases as POOR rises from 10% to 25%.

The predictions are relatively stable for k=10 and k=15, while k=5 gives somewhat different predictions (873.54), particularly at POOR = 10%. The standard errors are also generally smaller for k=10 and k=15. Therefore, k=10 or k=15 may provide a reasonable balance between local sensitivity and stability.